In [8]:
"""
Testing the dataloader EM property based on synthetic labels.
"""

'\nTesting the dataloader EM property based on synthetic labels.\n'

In [9]:
import sys
import torch
from collections import Counter
from equilibration_sampler import EquilibrationSampler

In [10]:
# ISIC 2019 training class distribution (Using numbers from proposal Table 1)
# Making synthetic data.
# CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
# These counts reflect the ~25K training images after UNK removal.

In [11]:
ISIC_CLASS_COUNTS = {
    0: 4522,   # MEL
    1: 12875,  # NV  (majority class)
    2: 3323,   # BCC
    3: 867,    # AK
    4: 2624,   # BKL
    5: 239,    # DF  (most extreme minority)
    6: 253,    # VASC
    7: 628,    # SCC
}
Num_Classes = 8

In [12]:
# Generating a flat label list to populate with the correct number of each class
labels = []
for Class, count in ISIC_CLASS_COUNTS.items():
    labels.extend([Class] * count)

# Distribution check:
for Class, count in ISIC_CLASS_COUNTS.items():
    names = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
    print(f"  class {Class} ({names[Class]:4s}): {count:5d} samples")
print()

  class 0 (MEL ):  4522 samples
  class 1 (NV  ): 12875 samples
  class 2 (BCC ):  3323 samples
  class 3 (AK  ):   867 samples
  class 4 (BKL ):  2624 samples
  class 5 (DF  ):   239 samples
  class 6 (VASC):   253 samples
  class 7 (SCC ):   628 samples



In [13]:
# Selecting batch sizes to test
Batch_size_test = [16, 24, 32, 48, 64]
Num_batch_check= 100

In [14]:
# Batch check logic
# Outer loop for batch size.
for batch_size in Batch_size_test:
   
   num_per_class = batch_size // Num_Classes

   sampler = EquilibrationSampler(
        labels=labels, 
        num_classes=Num_Classes,
        batch_size=batch_size,
        shuffle=True,
        seed = 42)
   
   # Utilizing _build_epoch_indices()
   epoch_indices = list(iter(sampler))
   effective_batch = num_per_class * Num_Classes
   
   print(f"Batch size: {batch_size} | num_per_batch: {num_per_class}")
   # Iterate over the Num_batch_check batches
   # Adding in a way to keep from slicing past the end of the synthetic data
   for batch_num in range(min(Num_batch_check, len(epoch_indices) // effective_batch)):
       start = batch_num * effective_batch
       ind_slice = epoch_indices[start:start + effective_batch]
       batch_labels = torch.tensor([labels[i] for i in ind_slice], dtype=torch.long)
       EquilibrationSampler.log_batch_class_distribution(
           labels=batch_labels,
           num_classes=Num_Classes,
           batch_idx=batch_num,
           log_every_n=1)
   print()
   sampler.class_summary()

Batch size: 16 | num_per_batch: 2

  Batch 0 class distribution (balanced=True):
    class  0:    2 samples
    class  1:    2 samples
    class  2:    2 samples
    class  3:    2 samples
    class  4:    2 samples
    class  5:    2 samples
    class  6:    2 samples
    class  7:    2 samples
  Total: 16 samples, expected Q=2 per class

  Batch 1 class distribution (balanced=True):
    class  0:    2 samples
    class  1:    2 samples
    class  2:    2 samples
    class  3:    2 samples
    class  4:    2 samples
    class  5:    2 samples
    class  6:    2 samples
    class  7:    2 samples
  Total: 16 samples, expected Q=2 per class

  Batch 2 class distribution (balanced=True):
    class  0:    2 samples
    class  1:    2 samples
    class  2:    2 samples
    class  3:    2 samples
    class  4:    2 samples
    class  5:    2 samples
    class  6:    2 samples
    class  7:    2 samples
  Total: 16 samples, expected Q=2 per class

  Batch 3 class distribution (balanced=True)

In [ ]:
# No warnings produced from method.  EM samples are balanced.

In [15]:
# Checking for DF oversampling concern

In [ ]:
for batch_size in Batch_size_test:

    num_per_class = batch_size // Num_Classes

    DF_count = ISIC_CLASS_COUNTS[5]

    sampler_temp = EquilibrationSampler(
        labels=labels, 
        num_classes=Num_Classes,
        batch_size=batch_size,
        seed = 42)
    
    DF_per_epoch = num_per_class * sampler_temp._num_batches
    # Count repeats
    DF_repeats_epoch = DF_per_epoch / DF_count
    print(f'''Batch size: {batch_size} | num per class: {num_per_class}
    DF seen per epoch (approximation): {DF_per_epoch}
    DF repeats per epoch: {DF_repeats_epoch} oversampling from {DF_count}
    ''')

Batch size: 16 | num per class: 2
    DF seen per epoch (approximation): 12876
    DF repeats per epoch: 53.8744769874477 oversampling from 239
    
Batch size: 24 | num per class: 3
    DF seen per epoch (approximation): 12876
    DF repeats per epoch: 53.8744769874477 oversampling from 239
    
Batch size: 32 | num per class: 4
    DF seen per epoch (approximation): 12876
    DF repeats per epoch: 53.8744769874477 oversampling from 239
    
Batch size: 48 | num per class: 6
    DF seen per epoch (approximation): 12876
    DF repeats per epoch: 53.8744769874477 oversampling from 239
    
Batch size: 64 | num per class: 8
    DF seen per epoch (approximation): 12880
    DF repeats per epoch: 53.89121338912134 oversampling from 239
    
